# FRCRN_SE_16K

In [1]:
import warnings
from pathlib import Path
from tqdm import tqdm
import os
import gc
import time
import numpy as np
import soundfile as sf
from scipy import signal
import torch
from clear_memory import clear_memory

warnings.filterwarnings('ignore')

In [2]:
input_dir_Pitt = Path('../ad_detection/data/raw/Pitt_origin')
output_dir_Pitt = Path('../ad_detection/data/denoised/Pitt-origin-FRCRN_SE')

control_files_Pitt = list((input_dir_Pitt / 'Control').glob('*.wav')) + list((input_dir_Pitt / 'Control').glob('*.mp3'))
dementia_files_Pitt = list((input_dir_Pitt / 'Dementia').glob('*.wav')) + list((input_dir_Pitt / 'Dementia').glob('*.mp3'))

input_dir_Lu = Path('../ad_detection/data/raw/Lu')
output_dir_Lu = Path('../ad_detection/data/denoised/Lu-FRCRN_SE')

control_files_Lu = list((input_dir_Lu / 'Control').glob('*.wav')) + list((input_dir_Lu / 'Control').glob('*.mp3'))
dementia_files_Lu = list((input_dir_Lu / 'Dementia').glob('*.wav')) + list((input_dir_Lu / 'Dementia').glob('*.mp3'))

## Load Model

In [3]:
from clearvoice import ClearVoice

model_name = 'FRCRN_SE_16K'
target_sr = 16000  # 目标采样率，必须与模型匹配

myClearVoice = ClearVoice(
    task='speech_enhancement',
    model_names=[model_name]
)

## Denoise Function

In [4]:
def denoise_audio(audio_path, model, target_sr=16000):
    """
    使用 ClearerVoice 进行语音降噪和增强
    
    Args:
        audio_path: 输入音频文件路径
        model: ClearVoice 模型实例
        target_sr: 目标采样率（16000 或 48000，取决于模型）
    
    Returns:
        denoised_audio: 降噪后的音频 numpy array
        sr: 采样率
    """
    # 加载音频
    audio, sr = sf.read(str(audio_path))
    
    # 如果是多声道，先转为单声道（在重采样之前）
    if len(audio.shape) == 2:
        # audio 形状是 (samples, channels)
        audio = np.mean(audio, axis=1)
    
    # 重采样到目标采样率（使用 scipy，更稳定）
    if sr != target_sr:
        # 计算目标样本数
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)
    
    # 确保是 float32 类型
    audio = audio.astype(np.float32)
    
    # 转换为 [batch, length] 格式
    audio = np.reshape(audio, [1, audio.shape[0]])
    
    # 应用 ClearVoice 降噪
    # 使用 torch.no_grad() 禁用梯度计算，节省显存
    with torch.no_grad():
        # online_write=False 表示返回 numpy 数组而不是直接写入文件
        output_wav = model(audio, online_write=False)
    
    # output_wav 形状: [batch, length]
    return output_wav[0, :], target_sr

In [5]:
def batch_denoise(files, output_subdir, model, target_sr, group_name):
    """
    批量降噪处理（每个文件前后都清理显存）
    
    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        model: ClearVoice 模型实例
        target_sr: 目标采样率
        group_name: 组名（用于显示进度）
    """
    # 创建输出目录
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    skip_count = 0
    fail_count = 0
    
    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        # 统一输出为 .wav 格式
        output_file = output_subdir / (audio_file.stem + '.wav')
        
        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue
        
        try:
            # ⚡ 处理前清理显存
            clear_memory()
            
            # 降噪
            denoised_audio, sr = denoise_audio(audio_file, model, target_sr)
            
            # 保存（16位整数格式）
            sf.write(str(output_file), denoised_audio, sr, subtype='PCM_16')
            success_count += 1
            
            # ⚡ 处理后立即清理显存
            del denoised_audio  # 删除大数组
            clear_memory()
            
        except Exception as e:
            fail_count += 1
            print(f"\nFailed: {audio_file.name}: {e}")
            # ⚡ 失败后也要清理显存
            clear_memory()
    
    # 打印统计信息
    print(f"\n{group_name} 处理完成:")
    print(f"成功: {success_count}")
    print(f"跳过: {skip_count}")
    print(f"失败: {fail_count}")
    print(f"总计: {len(files)}")

## Pitt Denoise

In [6]:
clear_memory()

batch_denoise(
    dementia_files_Pitt,
    output_dir_Pitt / 'Dementia',
    myClearVoice,
    target_sr=target_sr,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files_Pitt,
    output_dir_Pitt / 'Control',
    myClearVoice,
    target_sr=target_sr,
    group_name='Control'
)

Processing Dementia:  10%|█         | 32/309 [01:10<10:40,  2.31s/it]


Failed: 033-1.mp3: Cannot interpret '2514600' as a data type


Processing Dementia:  11%|█▏        | 35/309 [01:18<10:38,  2.33s/it]


Failed: 046-2.mp3: Cannot interpret '2048136' as a data type


Processing Dementia:  18%|█▊        | 57/309 [02:18<11:08,  2.65s/it]

...


Failed: 124-1.mp3: Cannot interpret '2193334' as a data type


Processing Control:  95%|█████████▌| 232/243 [08:46<00:23,  2.10s/it]


Failed: 225-2.mp3: Cannot interpret '2142168' as a data type


Processing Control: 100%|██████████| 243/243 [09:14<00:00,  2.28s/it]


Control 处理完成:
成功: 236
跳过: 0
失败: 7
总计: 243


## Lu Denoise

In [7]:
# clear_memory()

# batch_denoise(
#     dementia_files_Lu,
#     output_dir_Lu / 'Dementia',
#     myClearVoice,
#     target_sr=target_sr,
#     group_name='Dementia'
# )

# clear_memory()

# batch_denoise(
#     control_files_Lu,
#     output_dir_Lu / 'Control',
#     myClearVoice,
#     target_sr=target_sr,
#     group_name='Control'
# )